In [ ]:
import os

os.chdir("./mcp-net/")

In [ ]:
!git clone https://github.com/Beckschen/TransUNet.git

Cloning into 'TransUNet'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 128 (delta 71), reused 50 (delta 50), pack-reused 43 (from 2)
Receiving objects: 100% (128/128), 46.99 KiB | 1.30 MiB/s, done.
Resolving deltas: 100% (71/71), done.


In [ ]:
!cd TransUNet && pip install -r requirements.txt
!cd TransUNet && wget https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz
!cd TransUNet && mkdir -p ./model/vit_checkpoint/imagenet21k
!cd TransUNet && mv R50+ViT-B_16.npz ./model/vit_checkpoint/imagenet21k/

In [ ]:
!python "acdc_baselines/prepare_acdc_for_transunet.py"

# Train the model

In [ ]:
os.chdir("TransUNet")

In [ ]:
# Train the model
!CUDA_VISIBLE_DEVICES=0 python train.py --dataset ACDC --vit_name R50-ViT-B_16 \
    --max_epochs 150 --base_lr 0.01 --img_size 224 --batch_size 24 --seed 5 --deterministic 1

In [ ]:
#TODO update the following block inside TransUNet/test.py, then run the next cell
# dataset_config = {
#         'Synapse': {
#             'Dataset': Synapse_dataset,
#             'volume_path': '../data/Synapse/test_vol_h5',
#             'list_dir': './lists/lists_Synapse',
#             'num_classes': 9,
#             'z_spacing': 1,
#         },
#         'ACDC': {
#              'Dataset': ACDC_dataset,
#              'volume_path': '../data/transUnet/test_vol_h5',
#              'list_dir': '../data/transUnet/lists_ACDC',
#              'num_classes': 4,
#              'z_spacing': 1,
#          }
#
#     }


In [ ]:
# run on hold out set
!python test.py --dataset ACDC --vit_name R50-ViT-B_16 --img_size 224 --max_epochs 150 --seed 5 --deterministic 1 --batch_size 24 --max_iterations 30000 --is_savenii


## Preparing MMs for TransUnet

In [ ]:
!python mms_baselines/prepare_mms_for_transunet.py

Loaded metadata CSV: /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/M&Ms/211230_M&Ms_Dataset_information_diagnosis_opendataset.csv
Columns found: ['Unnamed: 0', 'External code', 'VendorName', 'Vendor', 'Centre', 'ED', 'ES', 'Age', 'Pathology', 'Sex', 'Height', 'Weight']

Saved 340 volume(s) to /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/M&Ms/mms_for_transunet/test_vol_h5
List file: /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/M&Ms/mms_for_transunet/lists_ACDC/test_vol.txt

Add to test.py's dataset_config:
  'MMS': {'Dataset': ACDC_dataset, 'volume_path': '/content/drive/MyDrive/Colab_Notebooks/mcp-net/data/M&Ms/mms_for_transunet/test_vol_h5', 'list_dir': '/content/drive/MyDrive/Colab_Notebooks/mcp-net/data/M&Ms/mms_for_transunet/lists_ACDC', 'num_classes': 4, 'z_spacing': 1}


In [ ]:
#TODO: inside TransUNet/test.py
# update the ACDC block with paths to the MMs dataset, then run the next cell
# dataset_config = {
#         'Synapse': {
#             'Dataset': Synapse_dataset,
#             'volume_path': '../data/Synapse/test_vol_h5',
#             'list_dir': './lists/lists_Synapse',
#             'num_classes': 9,
#             'z_spacing': 1,
#         },
#         'ACDC': {
#              'Dataset': ACDC_dataset,
#              'volume_path': '../data/transUnet/test_vol_h5',
#              'list_dir': '../data/transUnet/lists_ACDC',
#              'num_classes': 4,
#              'z_spacing': 1,
#          },
#          'MMS': {
#              'Dataset': ACDC_dataset, 
#              'volume_path': 'data/M&Ms/mms_for_transunet/test_vol_h5', 
#              'list_dir': 'data/M&Ms/mms_for_transunet/lists_ACDC', 
#              'num_classes': 4, 
#              'z_spacing': 1
#           }

#
#     }

In [ ]:
# prediction on MMs
!python test.py --dataset ACDC --vit_name R50-ViT-B_16 --img_size 224 --max_epochs 150 --seed 5 --deterministic 1 --batch_size 24 --max_iterations 30000 --is_savenii


## Evaluating performance

In [ ]:
import time

import numpy as np
import torch
from torch.utils.flop_counter import FlopCounterMode



def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_flops(model, input_shape, device):
    
    dummy = torch.randn(1, *input_shape).to(device)
    model.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fcm:
        model(dummy)
    return fcm.get_total_flops()


def benchmark_inference_time(model, input_shape, device, n_warmup=10, n_runs=50):
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(dummy)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - start)

    return np.mean(times), np.std(times)


def benchmark_peak_memory(model, input_shape, device):
    if device.type != "cuda":
        return None  # peak memory tracking only meaningful on GPU here
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)
    torch.cuda.reset_peak_memory_stats(device)
    with torch.no_grad():
        _ = model(dummy)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB


def benchmark_pytorch_model(model, input_shape, model_name, device=None, slices_per_volume=10):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    n_params = count_params(model)
    flops = count_flops(model, input_shape, device)
    mean_time_per_slice, std_time_per_slice = benchmark_inference_time(model, input_shape, device)
    peak_mem_mb = benchmark_peak_memory(model, input_shape, device)

    print(f"=== {model_name} ===")
    print(f"Parameters: {n_params:,}")
    print(f"FLOPs: {flops/1e9:.3f} GFLOPs")
    print(f"Inference time per slice: {mean_time_per_slice*1000:.2f} ± {std_time_per_slice*1000:.2f} ms")
    print(f"Extrapolated time per volume ({slices_per_volume} slices, sequential): "
          f"{mean_time_per_slice*slices_per_volume*1000:.2f} ms")
    if peak_mem_mb is not None:
        print(f"Peak GPU memory: {peak_mem_mb:.1f} MB")
    else:
        print("Peak GPU memory: N/A (running on CPU)")
    print(f"Device: {device}")
    print()

    return {
        "model_name": model_name, "params": n_params, "flops": flops,
        "mean_time_per_slice_ms": mean_time_per_slice * 1000,
        "std_time_per_slice_ms": std_time_per_slice * 1000,
        "time_per_volume_ms": mean_time_per_slice * slices_per_volume * 1000,
        "peak_memory_mb": peak_mem_mb, "device": str(device),
    }

In [ ]:
import os

os.chdir('TransUNet')

In [ ]:
from networks.vit_seg_modeling import VisionTransformer as ViT_seg
from networks.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg
import torch

vit_name = "R50-ViT-B_16"   # whatever you actually trained with
num_classes = 4
img_size = 224
n_skip = 3                   # whatever --n_skip value you trained with
vit_patches_size = 16        # whatever --vit_patches_size you trained with

config_vit = CONFIGS_ViT_seg[vit_name]
config_vit.n_classes = num_classes
config_vit.n_skip = n_skip
if vit_name.find('R50') != -1:
    config_vit.patches.grid = (int(img_size / vit_patches_size), int(img_size / vit_patches_size))

model = ViT_seg(config_vit, img_size=img_size, num_classes=num_classes)

checkpoint = torch.load("./model/TU_ACDC224/TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224_s5/epoch_149.pth", map_location="cpu")
model.load_state_dict(checkpoint)

<All keys matched successfully>

In [ ]:
from networks.vit_seg_modeling import VisionTransformer

benchmark_pytorch_model(model, (1, 224, 224), "TransUNet")

=== TransUNet ===
Parameters: 105,276,356
FLOPs: 58.379 GFLOPs
Inference time per slice: 32.40 ± 2.13 ms
Extrapolated time per volume (10 slices, sequential): 323.97 ms
Peak GPU memory: 1263.2 MB
Device: cuda



{'model_name': 'TransUNet',
 'params': 105276356,
 'flops': 58379206656,
 'mean_time_per_slice_ms': np.float64(32.39667663999171),
 'std_time_per_slice_ms': np.float64(2.1337166562727248),
 'time_per_volume_ms': np.float64(323.9667663999171),
 'peak_memory_mb': 1263.197265625,
 'device': 'cuda'}